# S6E9: Income Needs 1782 Bins, LightGBM Gives 255

For every numeric column in this competition, I measured how much AUC the column carries as a
function of how finely it is cut. Squeeze one column into 1, 2, 4, 8 and so on up to thousands of
cells, keep every other column at full resolution, refit the same fixed LightGBM on the same five
frozen folds, and read the AUC. That makes 44 coarsened points across seven columns, each one a
full five-fold fit. Then I asked the question a reader actually cares about: does a strong model
already have what these curves find?

Four things came out of it.

1. **Resolution is a one-column effect.** Of the seven numeric columns, only `Annual_Income_USD`
   keeps paying as you give it more cells. Its curve does not come within two standard errors of
   its best point until **1,782 cells**. `Daily_Commute_km` is the only other column with more than
   255 distinct values, and inside the model it carries no signal I can measure at any
   resolution: holding it constant costs 1.33 SE, under the bar. `Environmental_Concern_Level`
   carries by far the most signal (holding it constant costs 0.092242) and needs all five of its
   levels, but five levels is not a resolution problem.
2. **LightGBM's default binning is a coarsening, and here it costs +0.001999.** By default
   LightGBM gives income 255 histogram bins. Opening the histogram so every distinct value gets
   its own bin moves the same model, with the same seed and the same 13 raw columns, from
   **0.941522 to 0.943521**. That is **5.72 standard errors**, higher in 5 of 5 folds, from
   configuration alone, with no new feature.
3. **The signal sits between a $10 grid and a $100 grid.** Rounding income to the nearest $10
   costs 0.39 SE. Rounding it to the nearest $100 costs 3.05 SE.
4. **A strong model already has it.** On top of the default-binned LightGBM, a cross-fitted
   correction on income's finest grid is worth +0.001906, which is most of what opening the bins
   bought. On top of a logistic blend of the 19 models in my
   [S6E9 OOF library](https://www.kaggle.com/datasets/dariushafshar/s6e9-golem-oof-library)
   (0.944743), the same correction is worth +0.000024, or +0.000047 by a leak-free estimate:
   0.08 and 0.16 of the blend's standard error.

**So, will this move your score?** If you run a gradient-boosted tree on the raw columns with
default binning, yes: this is the cheapest measurable gain I found on this board, and it is one
parameter. If your model already carries target encoding, quantisation digits or frequency
features of income, it almost certainly has this already and this notebook will not move your
score. I measured that rather than assuming it, in Part 5.

The theory, the knee rule, the predicted shape of every curve and three falsifiers were written
down before any AUC here was computed. Part 6 scores them. None of the three falsifiers fired.
Competition: [Playground Series S6E9](https://www.kaggle.com/competitions/playground-series-s6e9).

## The cell you edit

Everything that defines the measurement is in this one dictionary. `GRID` is the set of cell
counts each column is squeezed into. `LGB` is the curve model, fixed before any number below was
computed and never tuned against one. `BAR_SE` decides what counts as a finding: a difference
smaller than two fold standard errors of the full-resolution model is reported, but it is not
called a finding.

To run the same measurement on a different board, change the `TARGET`, point `GRID` at the cell
counts you care about, and leave the rest alone.

In [1]:
CFG = {
    "TARGET": "Will_Buy_EV",
    "ID_COL": "id",
    "POSITIVE": "Yes",
    "SEED": 42,
    "N_SPLITS": 5,

    # --- the resolution grid ------------------------------------------------
    # Equal-mass quantile cells, ties never split. A point is dropped when the
    # column has fewer distinct values than it, or when it reproduces a coarser
    # point's partition exactly. Full resolution is always added at the end.
    "GRID": (1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 8192),
    # Rounding instead of quantile cells, for the widest column. $1 is its
    # native step (every value is a whole dollar).
    "INCOME_STEPS": (5, 10, 100, 1000),
    "COMMUTE_STEPS": (0.5, 1, 5, 10),     # single-column instrument only

    # --- the curve model: fixed, never tuned --------------------------------
    "LGB": dict(learning_rate=0.1, n_estimators=300, num_leaves=63,
                min_child_samples=40, subsample=0.8, subsample_freq=1,
                colsample_bytree=0.8, reg_lambda=1.0),
    "THREADS": 4,                 # Kaggle CPU; fixed so reruns are bit-stable
    "JITTER_SEEDS": (7, 2026),    # two extra seeds, to price the model's own jitter

    # --- the single-column instrument ---------------------------------------
    "LOOKUP_SMOOTH": 10.0,        # pseudo-counts toward the fold prior

    # --- the on-top probe ----------------------------------------------------
    "PROBE_LAMBDAS": (1.0, 10.0, 100.0),   # ridge on each cell's offset, Fisher-information units
    "PROBE_COARSE_CELLS": 32,     # coarse-grid control for the wide columns
    "PROBE_NULL_DRAWS": 3,

    # --- the noise bar -------------------------------------------------------
    "BAR_SE": 2.0,                # a difference below 2 SE is not a finding
}

---

## Part 1: The frame and the frozen folds

Every number in this notebook is out-of-fold on one partition:
`StratifiedKFold(5, shuffle=True, random_state=42)` over `train.csv` in file order. I do not
regenerate it and hope. I read it from the
[S6E9 OOF library](https://www.kaggle.com/datasets/dariushafshar/s6e9-golem-oof-library), where
it ships as `folds_seed42.npy` next to the out-of-fold predictions of every library member, and
then check it against its own recipe. That matters in Part 5, where those predictions are the
strong model. They are only comparable with anything measured here if the folds are the same
rows, and the check below says they are, on 668,665 of 668,665 rows.

In [2]:
import os, gc, json, math, time, glob, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
T0 = time.time()

ROOT = os.environ.get("SMOKE_ROOT", "/kaggle/input")
WORK = os.environ.get("SMOKE_WORK", "/kaggle/working")

def find(name):
    for root, _, files in os.walk(ROOT):
        if name in files:
            return os.path.join(root, name)
    raise FileNotFoundError(name)

TARGET, ID = CFG["TARGET"], CFG["ID_COL"]
train = pd.read_csv(find("train.csv"))
test = pd.read_csv(find("test.csv"))
LIB = os.path.dirname(find("folds_seed42.npy"))     # the mounted OOF library
N_FULL = len(train)

SMOKE_ROWS = int(os.environ.get("SMOKE_ROWS", "0"))
KEEP = np.arange(N_FULL)
if SMOKE_ROWS:
    KEEP = np.sort(np.random.default_rng(0).choice(N_FULL, SMOKE_ROWS, replace=False))
    train = train.iloc[KEEP].reset_index(drop=True)
    print(f"SMOKE_ROWS={SMOKE_ROWS}: train subsampled. Every number below is a "
          f"small-sample number and no title may be frozen from it.\n")

y = (train[TARGET].astype(str).str.strip() == CFG["POSITIVE"]).astype(np.int8).to_numpy()
yf = y.astype(np.float64)
N = len(train)
FEATS = [c for c in train.columns if c not in (ID, TARGET)]
CATEG = [c for c in FEATS if not pd.api.types.is_numeric_dtype(train[c])]
NUM = [c for c in FEATS if c not in CATEG]

FOLDS = np.load(os.path.join(LIB, "folds_seed42.npy"))[KEEP].astype(np.int8)
K = CFG["N_SPLITS"]
TR_I = [np.where(FOLDS != k)[0] for k in range(K)]
VA_I = [np.where(FOLDS == k)[0] for k in range(K)]
PRIOR = [float(yf[i].mean()) for i in TR_I]

print(f"  train rows            {N:>10,}")
print(f"  test rows             {len(test):>10,}")
print(f"  feature columns       {len(FEATS):>10}   ({len(NUM)} numeric, {len(CATEG)} not)")
print(f"  positive rate         {y.mean():>10.6f}")
print(f"  library mounted at    {LIB}")
print(f"  fold sizes            {np.bincount(FOLDS).tolist()}")

if not SMOKE_ROWS:
    from sklearn.model_selection import StratifiedKFold
    regen = np.full(N, -1, np.int8)
    for k, (_, va) in enumerate(StratifiedKFold(K, shuffle=True, random_state=CFG["SEED"])
                                .split(np.zeros(N), y)):
        regen[va] = k
    same = bool((regen == FOLDS).all())
    print(f"\nThe library's folds_seed42.npy against a fresh StratifiedKFold(5, shuffle=True, "
          f"random_state=42)\nover train.csv in file order: identical on "
          f"{int((regen == FOLDS).sum()):,} of {N:,} rows.")
    assert same, "fold file does not match its own recipe"

### Which columns can be coarsened, and what LightGBM already does to them

Seven of the thirteen columns are numeric. Resolution is a question about them only: a
categorical with three levels has no finer grid to find.

The cell below also asks LightGBM how many histogram bins it builds for each numeric column. A
gradient-boosted tree never sees a raw value. It sees the bin the value falls into, and by
default there are at most 255 of them. For five of the seven columns that is no constraint at
all. For the two wide ones it is a coarsening, applied silently before the first tree is grown:
`Annual_Income_USD` has 13,214 distinct values and gets 255 bins, and `Daily_Commute_km` has 805
and also gets 255.

The curve model in this notebook opens the histogram so that every distinct value gets its own
bin. Otherwise every curve would flatten at 255 by construction and I would be measuring
LightGBM's defaults instead of the data.

In [3]:
import lightgbm as lgb

DISTINCT = {c: int(train[c].nunique()) for c in NUM}
STEP = {c: float(np.diff(np.unique(train[c].to_numpy(float))).min()) for c in NUM}

X_FULL = train[FEATS].copy()
for c in CATEG:
    X_FULL[c] = X_FULL[c].astype("category")
X_TEST = test[FEATS].copy()
for c in CATEG:
    X_TEST[c] = pd.Categorical(X_TEST[c], categories=X_FULL[c].cat.categories)

OPEN_BINS = dict(max_bin=max(DISTINCT.values()) + 1, min_data_in_bin=1,
                 bin_construct_sample_cnt=max(N, 200000) + 1)

def lgb_bins(extra):
    ds = lgb.Dataset(X_FULL, label=yf, params=dict(verbose=-1, **extra)).construct()
    return {c: ds.feature_num_bin(FEATS.index(c)) for c in NUM}

BINS_DEFAULT = lgb_bins({})
BINS_OPEN = lgb_bins(OPEN_BINS)

print(f"{'numeric column':<30}{'distinct':>9}{'native step':>13}"
      f"{'LightGBM bins':>15}{'opened bins':>13}")
print("-" * 80)
for c in NUM:
    flag = "   <- default binning coarsens it" if BINS_DEFAULT[c] < DISTINCT[c] else ""
    print(f"{c:<30}{DISTINCT[c]:>9,}{STEP[c]:>13g}{BINS_DEFAULT[c]:>15,}{BINS_OPEN[c]:>13,}{flag}")
print("-" * 80)
print("LightGBM bins = what a default LightGBM builds (max_bin 255). Opened bins = what the")
print(f"curve model below builds: max_bin {OPEN_BINS['max_bin']:,}, min_data_in_bin 1, every row used to")
print("place the bin edges. Columns that never take the value 0 show one bin more than their")
print("distinct count, because LightGBM always keeps a bin for zero.")

def cells_of(codes):
    return int(len(np.unique(codes)))

def quantile_codes(x, r):
    if r == 1:
        return np.zeros(len(x), np.int32)
    return pd.qcut(x, r, labels=False, duplicates="drop").astype(np.int32)

def full_codes(x):
    return np.unique(x, return_inverse=True)[1].astype(np.int32)

def round_codes(x, step):
    return full_codes(np.round(x / step + 1e-9))

GRIDPTS = {}
for c in NUM:
    x = train[c].to_numpy(float)
    pts, seen = [], set()
    for r in CFG["GRID"]:
        if r >= DISTINCT[c]:
            break
        codes = quantile_codes(x, r)
        key = hash(codes.tobytes())
        if key in seen:
            continue
        seen.add(key)
        pts.append({"asked": r, "cells": cells_of(codes), "codes": codes})
    pts.append({"asked": "full", "cells": DISTINCT[c], "codes": full_codes(x)})
    GRIDPTS[c] = pts

print()
print("The resolution grid, in cells actually occupied (asked-for count in brackets when it differs):")
for c in NUM:
    s = ", ".join(f"{p['cells']:,}" + (f" [{p['asked']}]" if p["asked"] not in ("full", p["cells"]) else "")
                  for p in GRIDPTS[c])
    print(f"  {c:<30}{s}")
N_POINTS = sum(len(v) - 1 for v in GRIDPTS.values())
print(f"\n{N_POINTS} coarsened points across {len(NUM)} columns, each a full 5-fold fit, "
      f"plus the shared full-resolution fit.")

---

## Part 2: The instrument, and its noise priced first

**The curve model.** One LightGBM with the fixed parameters in the config cell, native
categoricals, and the histogram opened to full resolution. For each numeric column and each grid
point, that column is replaced by its equal-mass quantile cell and fed to the model as an ordinal
code. Ties are never split, and the cell edges come from the whole training frame without looking
at a label. **Every other column stays at full resolution.** The fit runs on all five folds, and
the fold-mean AUC is the curve's value at that point.

**The noise, before anything is measured with it.** The unit for everything below is the fold
standard error of the full-resolution model, $\mathrm{sd}(\text{fold AUCs})/\sqrt{5}$. A
difference under two of them is not a finding. Two extra seeds of the same model show what the
instrument does when nothing at all has changed.

In [4]:
from sklearn.metrics import roc_auc_score

LGB_PARAMS = dict(objective="binary", verbose=-1, n_jobs=CFG["THREADS"],
                  deterministic=True, force_row_wise=True, **CFG["LGB"])

def fold_aucs(score):
    return np.array([roc_auc_score(y[v], score[v]) for v in VA_I])

def run_lgb(X, seed=CFG["SEED"], open_bins=True, want_test=False):
    """5-fold OOF on the frozen folds. Returns fold AUCs, OOF vector, test mean."""
    params = dict(LGB_PARAMS, random_state=seed, **(OPEN_BINS if open_bins else {}))
    oof = np.zeros(N)
    tst = np.zeros(len(X_TEST)) if want_test else None
    for k in range(K):
        m = lgb.LGBMClassifier(**params).fit(X.iloc[TR_I[k]], y[TR_I[k]])
        oof[VA_I[k]] = m.predict_proba(X.iloc[VA_I[k]])[:, 1]
        if want_test:
            tst += m.predict_proba(X_TEST)[:, 1] / K
        del m
    gc.collect()
    return fold_aucs(oof), oof, tst

def se(fa):
    return float(np.std(fa, ddof=1) / math.sqrt(len(fa)))

t = time.time()
FA_FULL, OOF_FULL, TEST_FULL = run_lgb(X_FULL, want_test=True)
T_FIT = time.time() - t
JIT = {CFG["SEED"]: FA_FULL}
for s in CFG["JITTER_SEEDS"]:
    JIT[s] = run_lgb(X_FULL, seed=s)[0]
FA_DEF, OOF_DEF, _ = run_lgb(X_FULL, open_bins=False)

SE_FULL = se(FA_FULL)
BAR = CFG["BAR_SE"] * SE_FULL
AUC_FULL = float(FA_FULL.mean())

print(f"{'model':<44}" + "".join(f"{'fold ' + str(k):>10}" for k in range(K))
      + f"{'mean':>11}{'fold SE':>10}")
print("-" * 115)
def row(lbl, fa):
    print(f"{lbl:<44}" + "".join(f"{a:>10.6f}" for a in fa) + f"{fa.mean():>11.6f}{se(fa):>10.6f}")
for s, fa in JIT.items():
    row(f"full resolution, seed {s}", fa)
row("default binning (max_bin 255), seed 42", FA_DEF)
print("-" * 115)
print(f"one 5-fold fit takes {T_FIT:.0f}s here (with test predictions)")
print()
print("THE INSTRUMENT'S NOISE, priced before anything is measured with it")
print(f"  fold SE of the full-resolution model      {SE_FULL:.6f}   <- the SE used everywhere below")
print(f"  the bar for a finding (2 SE)              {BAR:.6f}")
seed_means = np.array([fa.mean() for fa in JIT.values()])
print(f"  spread of the fold-mean AUC across 3 seeds {seed_means.max() - seed_means.min():.6f}"
      f"   (sd {seed_means.std(ddof=1):.6f})")
pj = [se(JIT[s] - FA_FULL) for s in CFG["JITTER_SEEDS"]]
print(f"  paired fold SE of a seed-vs-seed difference {max(pj):.6f} (larger of the two)")
print(f"  pooled OOF AUC of the full-resolution model {roc_auc_score(y, OOF_FULL):.6f}")
print()
print(f"Changing nothing but the seed moves the fold-mean by at most "
      f"{seed_means.max() - seed_means.min():.6f}, which is "
      f"{(seed_means.max() - seed_means.min()) / SE_FULL:.2f} SE.")

The fold SE is 0.000349, so the bar for a finding is 0.000699. Changing nothing but the seed
moves the fold-mean by 0.000073, which is 0.21 SE. The instrument's own jitter is a fifth of the
unit it is priced in, which is what you want from an instrument.

### What the default binning costs, before a single curve

The table above already holds the notebook's most practical number. The default-binned model and
the full-resolution model are the same LightGBM on the same raw columns, with the same seed and
folds. The only difference is the histogram. The cell below prices that difference.

In [5]:
d = FA_FULL - FA_DEF
print("Default binning against opened binning: same model, same seed, same folds, same raw columns")
print(f"  full resolution    {FA_FULL.mean():.6f}")
print(f"  default max_bin    {FA_DEF.mean():.6f}")
print(f"  difference         {d.mean():+.6f}   = {d.mean() / SE_FULL:+.2f} SE   "
      f"(paired fold SE {se(d):.6f}, higher in {int((d > 0).sum())} of {K} folds)")
DEF_COST = float(d.mean())

It is +0.001999, or 5.72 SE, with a paired fold SE of 0.000083, and it is higher in 5 of 5 folds.
That is not subtle. The curves below say where it comes from.

---

## Part 3: The curves

Each numeric column is coarsened on its own while the other twelve columns stay exactly as they
were. The first table for each column is the model's AUC at every grid point, the difference from
full resolution, that difference in SE, and the paired fold SE of the difference. The paired SE is
there for information. The bar for a finding is two fold SEs of the level, which is stricter.

After the model's curves comes a second, much cheaper instrument: the column **alone**, as a
cross-fitted smoothed target rate per cell. It shows how much signal a column carries on its own at
each resolution. Where the two disagree, the difference is what the rest of the frame already
knows.

In [6]:
CURVE = {}
t_all = time.time()
for c in NUM:
    t = time.time()
    orig = X_FULL[c].copy()
    rows = []
    for p in GRIDPTS[c]:
        if p["asked"] == "full":
            fa = FA_FULL
        else:
            X_FULL[c] = p["codes"]
            fa = run_lgb(X_FULL)[0]
        rows.append({"cells": p["cells"], "asked": p["asked"], "fa": fa,
                     "mean": float(fa.mean()), "d": float((fa - FA_FULL).mean()),
                     "d_se": se(fa - FA_FULL)})
    X_FULL[c] = orig
    CURVE[c] = rows
    print(f"  {c:<30}{len(rows) - 1:>3} coarsened fits   {time.time() - t:6.0f}s", flush=True)
print(f"all curves: {time.time() - t_all:.0f}s\n")

for c in NUM:
    print(f"{c}  ({DISTINCT[c]:,} distinct values, native step {STEP[c]:g})")
    print(f"  {'cells':>7}{'fold-mean AUC':>15}{'vs full':>11}{'in SE':>8}{'paired SE':>11}")
    for r in CURVE[c]:
        tag = "   full" if r["asked"] == "full" else ""
        print(f"  {r['cells']:>7,}{r['mean']:>15.6f}{r['d']:>+11.6f}{r['d'] / SE_FULL:>8.2f}"
              f"{r['d_se']:>11.6f}{tag}")
    print()

In [7]:
def lookup_alone(codes, smooth=CFG["LOOKUP_SMOOTH"]):
    """Cross-fitted smoothed target rate per cell, the column alone. Fold AUCs."""
    ncell = int(codes.max()) + 1
    fa = np.zeros(K)
    for k in range(K):
        tr, va = TR_I[k], VA_I[k]
        cnt = np.bincount(codes[tr], minlength=ncell).astype(np.float64)
        pos = np.bincount(codes[tr], weights=yf[tr], minlength=ncell)
        rate = (pos + smooth * PRIOR[k]) / (cnt + smooth)
        fa[k] = roc_auc_score(y[va], rate[codes[va]])
    return fa

t = time.time()
ALONE = {}
for c in NUM:
    rows = []
    for p in GRIDPTS[c]:
        if p["cells"] == 1:
            fa = np.full(K, 0.5)
        else:
            fa = lookup_alone(p["codes"])
        rows.append({"cells": p["cells"], "asked": p["asked"], "fa": fa, "mean": float(fa.mean())})
    ALONE[c] = rows
print(f"single-column lookups, every column at every grid point ({time.time() - t:.0f}s)\n")
print(f"{'column':<30}{'AUC alone at each grid point, coarse to full':}")
for c in NUM:
    s = "  ".join(f"{r['cells']:,}:{r['mean']:.4f}" for r in ALONE[c])
    print(f"  {c:<28}{s}")

In [8]:
def knee(rows, se_ref, bar=CFG["BAR_SE"]):
    """Coarsest point of the plateau that holds the curve's best point.

    Every point from the knee up to the best point is within bar*SE of the best.
    Also returns how far the curve turns DOWN past its best point, if it does."""
    means = [r["mean"] for r in rows]
    b = int(np.argmax(means))
    lo = means[b] - bar * se_ref
    i = b
    while i > 0 and means[i - 1] >= lo:
        i -= 1
    turn = means[b] - min(means[b:])
    return i, b, turn

SUMMARY = []
for c in NUM:
    lg, al = CURVE[c], ALONE[c]
    i, b, turn = knee(lg, SE_FULL)
    se_al = se(al[-1]["fa"])
    ia, ba, turna = knee(al, se_al)
    SUMMARY.append({
        "col": c, "distinct": DISTINCT[c],
        "value": lg[-1]["mean"] - lg[0]["mean"],
        "knee": lg[i]["cells"], "knee_auc": lg[i]["mean"], "best_cells": lg[b]["cells"],
        "knee_loss": lg[b]["mean"] - lg[i]["mean"], "turn": turn,
        "below_knee": (lg[b]["mean"] - lg[i - 1]["mean"]) if i > 0 else None,
        "below_cells": lg[i - 1]["cells"] if i > 0 else None,
        "knee_alone": al[ia]["cells"], "best_alone": al[ba]["mean"],
        "best_alone_cells": al[ba]["cells"], "se_alone": se_al, "turn_alone": turna,
        "has_signal": (lg[-1]["mean"] - lg[0]["mean"]) > BAR,
    })

print(f"SE = {SE_FULL:.6f}; a difference counts only above 2 SE = {BAR:.6f}\n")
print(f"{'column':<30}{'distinct':>9}{'column value':>14}{'in SE':>8}"
      f"{'knee':>7}{'AUC at knee':>13}{'next coarser':>14}{'knee alone':>12}{'best alone':>12}")
print("-" * 119)
for s in SUMMARY:
    nxt = "-" if s["below_cells"] is None else f"-{s['below_knee']:.6f}"
    print(f"{s['col']:<30}{s['distinct']:>9,}{s['value']:>+14.6f}{s['value'] / SE_FULL:>8.1f}"
          f"{s['knee']:>7,}{s['knee_auc']:>13.6f}{nxt:>14}{s['knee_alone']:>12,}{s['best_alone']:>12.6f}")
print("-" * 119)
print("column value = full-resolution AUC minus the AUC with the column held constant.")
print("knee = coarsest cell count from which the curve stays within 2 SE of its best point.")
print("next coarser = how far below the best point the grid point just coarser than the knee sits.")
print()
for s in SUMMARY:
    if not s["has_signal"]:
        print(f"  {s['col']}: removing it entirely costs {s['value']:+.6f} "
              f"({s['value'] / SE_FULL:.2f} SE). No measurable signal at any resolution.")
for s in SUMMARY:
    if s["turn"] > BAR:
        print(f"  {s['col']}: the LightGBM curve turns down past its best point by {s['turn']:.6f}.")
    if s["turn_alone"] > CFG["BAR_SE"] * s["se_alone"]:
        print(f"  {s['col']}: the single-column curve peaks at {s['best_alone_cells']:,} cells and "
              f"falls {s['turn_alone']:.6f} by full resolution.")

In [9]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

fig, axes = plt.subplots(2, 4, figsize=(15, 7.6))
axes = axes.ravel()
order = sorted(SUMMARY, key=lambda s: -s["distinct"])
for ax, s in zip(axes, order):
    c = s["col"]
    notes = []
    for rows, lbl, colr, ls, se_ref in ((CURVE[c], "all 13 columns, LightGBM", "#1a1a1a", "-", SE_FULL),
                                        (ALONE[c], "this column alone, lookup", "#b03030", "--",
                                         s["se_alone"])):
        lo, hi = rows[0]["mean"], max(r["mean"] for r in rows)
        if hi - lo <= CFG["BAR_SE"] * se_ref:
            notes.append(lbl.split(",")[0] + ":\nno signal above 2 SE,\nnot drawn")
            continue
        xs = [r["cells"] for r in rows]
        ys = [(r["mean"] - lo) / (hi - lo) for r in rows]
        ax.plot(xs, ys, marker="o", ms=3.5, lw=1.5, ls=ls, color=colr, label=lbl)
        if ls == "-":
            ax.axhspan(1 - CFG["BAR_SE"] * se_ref / (hi - lo), 1.02, color="#2a6", alpha=0.10, lw=0)
            ax.axvline(s["knee"], color="#2a6", lw=1.3, ls=":")
    if s["distinct"] > 255:
        ax.axvline(255, color="#888", lw=0.9, ls="-.")
        ax.text(255, 0.03, " 255 = LightGBM\n default max_bin", fontsize=7.5, color="#666")
    if notes:
        at_top = s["distinct"] > 255      # keep clear of the max_bin label at the bottom right
        ax.text(0.04 if at_top else 0.97, 0.96 if at_top else 0.04, "\n".join(notes),
                transform=ax.transAxes, fontsize=8, color="#555",
                ha="left" if at_top else "right", va="top" if at_top else "bottom")
    ax.set_xscale("log", base=2)
    ax.set_ylim(-0.05, 1.08)
    knee_txt = (f"LightGBM knee at {s['knee']:,} of {s['distinct']:,} cells" if s["has_signal"]
                else f"{s['distinct']:,} cells, LightGBM value below 2 SE")
    ax.set_title(f"{c}\nLightGBM value {s['value']:+.5f} AUC ({s['value'] / SE_FULL:.1f} SE)\n{knee_txt}",
                 fontsize=8.8)
    ax.set_xlabel("cells the column is given (log scale)", fontsize=8.5)
    ax.set_ylabel("share of the column's signal captured", fontsize=8.5)
    ax.grid(alpha=0.25, lw=0.5)
    ax.tick_params(labelsize=8)
axes[-1].axis("off")
handles = [plt.Line2D([], [], color="#1a1a1a", marker="o", ms=3.5, lw=1.5,
                      label="all 13 columns, LightGBM (0 = column held constant, 1 = best point)"),
           plt.Line2D([], [], color="#b03030", marker="o", ms=3.5, lw=1.5, ls="--",
                      label="this column alone, lookup (0 = AUC 0.5, 1 = best point)"),
           plt.Line2D([], [], color="#2a6", ls=":", lw=1.3, label="knee of the LightGBM curve"),
           Patch(color="#2a6", alpha=0.18, label="within 2 SE of the LightGBM best point")]
axes[-1].legend(handles=handles, loc="center", fontsize=8.6, frameon=False)
fig.suptitle("Resolution curves: each numeric column coarsened alone, every other column at full resolution",
             fontsize=11.5)
fig.tight_layout()
plt.show()

### Reading the curves

**Income is the only numeric column with a resolution curve worth the name.** Held constant, it
costs 0.016775, or 48.03 SE. At 233 cells it is still −0.001991 below full resolution, which is
almost exactly what the default binning cost in Part 2 (+0.001999). That is close to the same
measurement made twice, because LightGBM's default histogram also cuts income into roughly
equal-count bins, 255 of them. The
curve then keeps climbing: 926 cells is −0.001047 (3.00 SE), and **1,782 cells** is the first
point within two SE of the best one, at −0.000620. It keeps creeping up after that, to −0.000157
at 4,808 cells, but every step past the knee is inside the noise bar.

On its own, income tells the same story more strongly. Its single-column AUC climbs to 0.712126
at 4,808 cells, and that is its knee. This is the per-value structure the
[income fine-structure notebook](https://www.kaggle.com/code/dariushafshar/s6e9-53-income-spikes-94-of-the-gain)
found by a different route: the generator attaches a propensity to individual income values, not
to a smooth function of income, so every merge of two neighbouring values throws some of it away.

**`Environmental_Concern_Level` is the strongest column on the board and has no resolution
question.** Held constant, it costs 0.092242, which is 264.10 SE. Merging its five levels into
four costs −0.000748 (2.14 SE), so the knee is at all five. Every level is needed, and there are
only five.

**Everything else is flat, and the reason is not the one I expected.** I predicted knees for
commute, age and the two charging-station counts. What came back instead is that **none of them
carries a signal the model can measure at any resolution.** Holding `Daily_Commute_km` constant
costs 0.000466 (1.33 SE). `Age` costs 0.000120 (0.34 SE). Holding the number of cars or either
charging-station count constant makes the model a hair *better*, at −0.000023, −0.000040 and
−0.000037, which is noise.

Commute deserves one honest sentence more. Its paired fold SE at one cell is 0.000049, so a paired
test would call that 0.000466 nonzero. The registered bar is two fold SEs of the level, 0.000699,
and commute does not clear it. I am reporting it and not calling it a finding.

**The single-column curves would have told you otherwise.** Alone, `Age` keeps gaining all the
way to its 45 levels (0.547015), and commute alone does not reach its plateau until 201 cells
(best 0.551410). A
resolution curve drawn on one column in isolation measures what that column knows. It does not
measure what it adds. Inside the model, the other twelve columns already carry what `Age` and
commute know, and their curves go flat.

The two charging-station counts are worth a note of their own. The
[interaction census](https://www.kaggle.com/code/dariushafshar/s6e9-4-of-13-columns-hold-the-interaction)
put `Charging_Stations_Near_Home` inside the one interaction block this generator has. Here,
holding it constant costs nothing measurable. Both can be true: this curve removes the column
from a model that still has the other three block members, and the census measured the block as
a whole.

---

## Part 4: Where the generator's grid sits

Quantile cells are one way to coarsen. Rounding to a step is the other, and it asks a more
physical question: at what dollar resolution does income's signal live? Every income value here
is a whole dollar, so $1 is the native grid. The cell below rounds income to $5, $10, $100 and
$1,000 and refits the same model, then puts each rounding next to the quantile grid point nearest
to it in cell count.

In [10]:
x_inc = train["Annual_Income_USD"].to_numpy(float)
x_com = train["Daily_Commute_km"].to_numpy(float)
ROUND = []
orig = X_FULL["Annual_Income_USD"].copy()
t = time.time()
for st in CFG["INCOME_STEPS"]:
    codes = round_codes(x_inc, st)
    X_FULL["Annual_Income_USD"] = codes
    fa = run_lgb(X_FULL)[0]
    ROUND.append({"col": "Annual_Income_USD", "step": st, "cells": cells_of(codes), "fa": fa,
                  "mean": float(fa.mean()), "d": float((fa - FA_FULL).mean()),
                  "d_se": se(fa - FA_FULL), "alone": float(lookup_alone(codes).mean())})
X_FULL["Annual_Income_USD"] = orig
print(f"income rounded to {len(CFG['INCOME_STEPS'])} steps, 5-fold fit each ({time.time() - t:.0f}s)\n")

inc_q = {r["cells"]: r for r in CURVE["Annual_Income_USD"]}
full_alone_inc = ALONE["Annual_Income_USD"][-1]["mean"]
print("Annual_Income_USD, rounded to a step instead of cut into quantile cells")
print(f"  {'step':>8}{'cells':>8}{'LightGBM AUC':>14}{'vs full':>11}{'in SE':>8}{'paired SE':>11}"
      f"{'alone':>10}")
print(f"  {'$1':>8}{DISTINCT['Annual_Income_USD']:>8,}{AUC_FULL:>14.6f}{0:>+11.6f}{0:>8.2f}{'':>11}"
      f"{full_alone_inc:>10.4f}   native: every value is a whole dollar")
for r in ROUND:
    print(f"  {'$' + format(r['step'], ','):>8}{r['cells']:>8,}{r['mean']:>14.6f}{r['d']:>+11.6f}"
          f"{r['d'] / SE_FULL:>8.2f}{r['d_se']:>11.6f}{r['alone']:>10.4f}")
print()
print("For comparison, the quantile cells nearest in count:")
for r in ROUND:
    near = min(CURVE["Annual_Income_USD"], key=lambda q: abs(math.log(q["cells"] / r["cells"])))
    print(f"  ${r['step']:<6,} {r['cells']:>6,} cells {r['mean']:.6f}   vs quantile "
          f"{near['cells']:>6,} cells {near['mean']:.6f}   ({r['mean'] - near['mean']:+.6f})")

print()
print("Daily_Commute_km, rounded (single-column lookup only; the LightGBM curve above already")
print("shows this column saturating)")
print(f"  {'step':>8}{'cells':>8}{'alone':>10}")
print(f"  {'0.1 km':>8}{DISTINCT['Daily_Commute_km']:>8,}{ALONE['Daily_Commute_km'][-1]['mean']:>10.4f}"
      f"   native step")
for st in CFG["COMMUTE_STEPS"]:
    codes = round_codes(x_com, st)
    print(f"  {str(st) + ' km':>8}{cells_of(codes):>8,}{lookup_alone(codes).mean():>10.4f}")

mult = {m: float(np.mean(np.round(x_inc) % m == 0)) for m in (5, 10, 100, 1000)}
print()
print("Where the income values sit on the dollar grid (share of rows on multiples of m, uniform = 1/m):")
for m, v in mult.items():
    print(f"  multiple of {m:>5,}: {v:.4f}  (uniform {1 / m:.4f}, ratio {v * m:.2f})")

**The signal lives between $10 and $100.** Rounding income to $5 costs 0.20 SE and rounding to
$10 costs 0.39 SE, both inside the noise. Rounding to $100 costs 3.05 SE, and to $1,000 it costs
7.74 SE. So a $10 grid keeps what the model can use, and a $100 grid does not.

**It is the number of cells, not where they fall.** At matched cell counts, rounding and quantile
cells agree to within 0.000102 on every row of the comparison. A reader who wants to coarsen
income for a lookup or an encoding can do it either way. What matters is keeping a few thousand
cells.

**The dollar grid is not uniform, and the heaping is a fingerprint.** A share of 0.2941 of the
rows sits on a multiple of $5, where a uniform grid would put 0.2000, and 0.1993 on a multiple of
$10. The
multiple-of-$1,000 share is almost all the hard floor at the bottom of the column, which the
[income fine-structure notebook](https://www.kaggle.com/code/dariushafshar/s6e9-53-income-spikes-94-of-the-gain)
documents. None of that heaping is where the signal is: rounding onto the $5 and $10 lattice
costs nothing measurable.

Commute, rounded, loses single-column signal steadily (0.5510 at its native 0.1 km, 0.5377 at
10 km), but Part 3 already showed that the model does not use it at any resolution.

---

## Part 5: Does a strong model already have it?

This is the section that decides whether any of the above is worth your afternoon. Three base
models, each an out-of-fold prediction on the same frozen folds:

- the **default-binned LightGBM** from Part 2, which is what most first notebooks on this board run;
- the **full-resolution LightGBM** from Part 2;
- a **blend of the OOF library**: a logistic regression on the logits of every library member,
  fitted on four folds and scored on the fifth.

In [11]:
from sklearn.linear_model import LogisticRegression

def logit(p):
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return np.log(p / (1 - p))

stems = sorted(os.path.basename(p)[4:-4] for p in glob.glob(os.path.join(LIB, "oof_*.npy")))
Z_LIB = np.column_stack([logit(np.load(os.path.join(LIB, f"oof_{s}.npy"))[KEEP]) for s in stems])
single = {s: float(fold_aucs(Z_LIB[:, j]).mean()) for j, s in enumerate(stems)}

OOF_BLEND = np.zeros(N)
for k in range(K):
    m = LogisticRegression(C=1.0, max_iter=2000).fit(Z_LIB[TR_I[k]], y[TR_I[k]])
    OOF_BLEND[VA_I[k]] = m.predict_proba(Z_LIB[VA_I[k]])[:, 1]
FA_BLEND = fold_aucs(OOF_BLEND)
best_s = max(single, key=single.get)

print(f"library members mounted: {len(stems)}  ({', '.join(stems)})")
print(f"  best single member      {best_s:<14}{single[best_s]:.6f}  fold-mean AUC")
print(f"  in-fold logistic blend  {'':<14}{FA_BLEND.mean():.6f}  fold-mean AUC, fold SE {se(FA_BLEND):.6f}")
print(f"  the curve model, full   {'':<14}{AUC_FULL:.6f}")
print(f"  the curve model, default binning {FA_DEF.mean():.6f}")
print()
print("The blend is a logistic regression on the logits of every member, fitted on four folds")
print("and scored on the fifth, so no row is scored by weights fitted on it.")

The blend scores 0.944743, above the best single member (h, 0.944513) and above both
LightGBMs.

**The probe.** For each base and each numeric column, I add the column's finest grid as a
cross-fitted correction and measure how much AUC it adds. Inside fold k, using only the rows
outside it, each cell of the column gets an offset: the sum of residuals in that cell divided by
the sum of $p(1-p)$ plus a ridge term $\lambda$. That is one Newton step of a per-cell intercept
on top of the base model, shrunk toward zero where a cell is thin. A cell the base model already
prices correctly gets an offset near zero. Then the rows of fold k are scored as base plus offset.

Three controls come with it:

- **a permutation null**: the same probe with the column's values shuffled across rows, so the
  cells are identical and carry no information;
- **a 32-cell control**: the same probe on a coarse quantile grid, which separates "the base
  misses the column's coarse shape" from "the base misses its fine structure";
- **a second, leak-free estimator.** The base prediction for a row in a training fold came from a
  model that saw fold k, so residuals estimated there are very slightly anti-correlated with fold
  k's labels, which biases the first estimator toward zero. The second one estimates offsets
  inside fold k, on one random half, and scores the other half, and the other way round. It has
  no such bias, but it estimates from about an eighth of the rows, so it is weaker. The two have opposite
  weaknesses, and both are printed.

**Two things about this probe changed after small-sample correctness runs and before any
full-size number existed, and both are in the pre-registration file with timestamps.** On a
60,000-row run the probe lost AUC even for a random key, so I added two more shrinkage levels
($\lambda$ = 10 and 100, best of the three reported, with the null put through the same
best-of-three so the selection is priced). The same run showed the probe reading a loss on a
five-level column with thousands of rows per cell while its null read zero, because a globally
overconfident base makes per-cell residuals track each cell's average score. So each base is now
recalibrated inside the fold by a one-dimensional logistic fit before the residuals are taken.
That fit is strictly monotone, so the base AUC does not move. Both changes make the falsifier
below easier to fire, not harder.

In [12]:
def probe(z, codes, lams=CFG["PROBE_LAMBDAS"]):
    """Per-fold AUC gain from one cross-fitted Newton step of per-cell offsets on top of z.

    Inside fold k, using only the rows outside it:
      1. recalibrate the base: zc = a + b*z, a 1-D logistic fit (strictly monotone,
         so the base AUC does not move). Without this, a model that is globally
         over- or under-confident produces per-cell residuals that merely track each
         cell's average score, and shifting whole cells re-ranks rows for nothing;
      2. offset_v = sum(y - p) / (sum p(1 - p) + lam) over the rows in cell v, p = sigmoid(zc).
    Then score = zc + offset on the rows of fold k. A cell the base already prices
    correctly gets an offset near zero; a sparse cell is shrunk toward zero.
    Returns an array (len(lams), K) of fold gains, one row per shrinkage level."""
    ncell = int(codes.max()) + 1
    g = np.zeros((len(lams), K))
    for k in range(K):
        tr, va = TR_I[k], VA_I[k]
        cal = PLATT[id(z)][k]
        zc = cal[0] + cal[1] * z
        p = 1 / (1 + np.exp(-zc[tr]))
        num = np.bincount(codes[tr], weights=yf[tr] - p, minlength=ncell)
        den = np.bincount(codes[tr], weights=p * (1 - p), minlength=ncell)
        base = roc_auc_score(y[va], z[va])
        for j, lam in enumerate(lams):
            off = num / (den + lam)
            g[j, k] = roc_auc_score(y[va], zc[va] + off[codes[va]]) - base
    return g

HALF = np.random.default_rng(CFG["SEED"] + 1).integers(0, 2, N).astype(np.int8)

def probe_halves(z, codes, lams=CFG["PROBE_LAMBDAS"]):
    """The same correction with no stacking leak: estimated INSIDE fold k.

    The base prediction for a row in fold k came from a model that never saw fold
    k, so residuals on one half of fold k carry no information about the other
    half's labels. Offsets from half A score half B and vice versa. Unbiased, but
    each offset is estimated from about a tenth of the rows the 4-fold version uses."""
    ncell = int(codes.max()) + 1
    g = np.zeros((len(lams), K))
    for k in range(K):
        va = VA_I[k]
        cal = PLATT[id(z)][k]
        zc = cal[0] + cal[1] * z[va]
        p = 1 / (1 + np.exp(-zc))
        cv, hv = codes[va], HALF[va]
        base = roc_auc_score(y[va], z[va])
        for j, lam in enumerate(lams):
            sc = zc.copy()
            for h in (0, 1):
                src, dst = hv == h, hv != h
                num = np.bincount(cv[src], weights=yf[va][src] - p[src], minlength=ncell)
                den = np.bincount(cv[src], weights=p[src] * (1 - p[src]), minlength=ncell)
                sc[dst] = zc[dst] + (num / (den + lam))[cv[dst]]
            g[j, k] = roc_auc_score(y[va], sc) - base
    return g

BASES = {"default-binned LightGBM": logit(OOF_DEF),
         "full-resolution LightGBM": logit(OOF_FULL),
         "library blend": logit(OOF_BLEND)}
BASE_SE = {"default-binned LightGBM": se(FA_DEF), "full-resolution LightGBM": SE_FULL,
           "library blend": se(FA_BLEND)}
LAMS = CFG["PROBE_LAMBDAS"]

PLATT = {}
print("In-fold recalibration of each base, a + b*z fitted on the four training folds:")
for bname, z in BASES.items():
    cal = []
    for k in range(K):
        lr = LogisticRegression(C=1e6, max_iter=1000).fit(z[TR_I[k], None], y[TR_I[k]])
        cal.append((float(lr.intercept_[0]), float(lr.coef_[0, 0])))
    PLATT[id(z)] = cal
    b = [c[1] for c in cal]
    print(f"  {bname:<28} slope b from {min(b):.4f} to {max(b):.4f}"
          + ("   (b < 1: the raw logits are overconfident)" if max(b) < 1 else ""))
print()
rng = np.random.default_rng(CFG["SEED"])

t = time.time()
ONTOP = []
for bname, z in BASES.items():
    for c in NUM:
        fine = GRIDPTS[c][-1]["codes"]
        rec = {"base": bname, "col": c, "cells": DISTINCT[c], "base_se": BASE_SE[bname]}
        for tag, fn in (("", probe), ("h_", probe_halves)):
            g = fn(z, fine)
            gm = g.mean(axis=1)
            j = int(np.argmax(gm))                   # best of the three shrinkage levels
            nulls = np.array([fn(z, fine[rng.permutation(N)]).mean(axis=1)
                              for _ in range(CFG["PROBE_NULL_DRAWS"])])   # draws x lambdas
            rec.update({tag + "by_lam": [float(v) for v in gm], tag + "lam": LAMS[j],
                        tag + "gain": float(gm[j]), tag + "se": se(g[j]),
                        tag + "wins": int((g[j] > 0).sum()),
                        tag + "null_best": float(nulls.max(axis=1).mean()),
                        tag + "null_max": float(nulls.max())})
        rec["coarse"] = None
        if DISTINCT[c] > CFG["PROBE_COARSE_CELLS"]:
            rec["coarse"] = float(probe(z, quantile_codes(train[c].to_numpy(float),
                                                          CFG["PROBE_COARSE_CELLS"])).mean(axis=1).max())
        rec["either"] = max(rec["gain"], rec["h_gain"])
        ONTOP.append(rec)
print(f"{len(ONTOP)} base x column pairs, two estimators, {len(LAMS)} shrinkage levels, "
      f"{CFG['PROBE_NULL_DRAWS']} permutation nulls each ({time.time() - t:.0f}s)\n")

for bname in BASES:
    print(f"ON TOP OF: {bname}   (its own fold SE {BASE_SE[bname]:.6f}; bar 2 SE = "
          f"{CFG['BAR_SE'] * BASE_SE[bname]:.6f})")
    print(f"  {'':<37}{'-- residuals from the 4 other folds --':^52}{'-- inside the fold, half vs half --':^42}")
    print(f"  {'column':<30}{'cells':>7}{'gain':>11}{'paired SE':>11}{'folds +':>9}{'in base SE':>11}"
          f"{'null':>10}{'gain':>11}{'paired SE':>11}{'in base SE':>11}{'null':>10}{'32-cell':>11}")
    for r in [r for r in ONTOP if r["base"] == bname]:
        cs = "-" if r["coarse"] is None else f"{r['coarse']:+.6f}"
        mark = "  <- above the bar" if r["either"] > CFG["BAR_SE"] * r["base_se"] else ""
        print(f"  {r['col']:<30}{r['cells']:>7,}{r['gain']:>+11.6f}{r['se']:>11.6f}{r['wins']:>6} of {K}"
              f"{r['gain'] / r['base_se']:>11.2f}{r['null_best']:>+10.6f}"
              f"{r['h_gain']:>+11.6f}{r['h_se']:>11.6f}{r['h_gain'] / r['base_se']:>11.2f}"
              f"{r['h_null_best']:>+10.6f}{cs:>11}{mark}")
    print()

print("Per shrinkage level, residuals from the 4 other folds (lambda = " + ", ".join(f"{l:g}" for l in LAMS) + "):")
for r in ONTOP:
    print(f"  {r['base']:<26}{r['col']:<30}" + "".join(f"{v:>+11.6f}" for v in r["by_lam"])
          + f"   best lambda {r['lam']:g}")
print()
all_null = np.array([r["null_max"] for r in ONTOP])
print(f"gain = the best of the three shrinkage levels; paired SE, folds + and in base SE refer to it.")
print(f"null = a random permutation of the column (same cells, no information), passed through the")
print(f"same best-of-three; mean of {CFG['PROBE_NULL_DRAWS']} draws. Largest null draw anywhere: {all_null.max():+.6f}.")
print("32-cell = the same probe, best of three, on a 32-cell quantile grid of that column; shown")
print("only for the two columns wide enough to have a coarse grid distinct from their native one.")

In [13]:
fig, ax = plt.subplots(figsize=(11, 4.4))
cols_order = [s["col"] for s in order]
width = 0.26
colors = {"default-binned LightGBM": "#9aa6b2", "full-resolution LightGBM": "#4a5a6a",
          "library blend": "#b03030"}
for j, bname in enumerate(BASES):
    rr = {r["col"]: r for r in ONTOP if r["base"] == bname}
    xs = np.arange(len(cols_order)) + (j - 1) * width
    ax.bar(xs, [rr[c]["gain"] for c in cols_order], width, color=colors[bname], label=bname,
           yerr=[CFG["BAR_SE"] * rr[c]["se"] for c in cols_order], capsize=2.5,
           error_kw=dict(lw=0.8, ecolor="#333"))
    ax.scatter(xs, [rr[c]["h_gain"] for c in cols_order], marker="D", s=14, color="#111", zorder=3,
               label="same base, estimated inside the fold (no stacking leak)" if j == 0 else None)
ax.axhline(BAR, color="#2a6", ls="--", lw=1.1,
           label=f"2 SE of the full-resolution model ({BAR:.5f})")
ax.axhline(0, color="#888", lw=0.8)
ax.set_xticks(np.arange(len(cols_order)))
ax.set_xticklabels([c.replace("_", "\n", 2) for c in cols_order], fontsize=8)
ax.set_ylabel("fold-mean AUC gain")
ax.set_title("Adding each column's finest grid as a cross-fitted correction on top of a model\n"
             "(bars: residuals from the other 4 folds, best of three shrinkage levels, error bars 2 paired fold SE;\n"
             "diamonds: the leak-free half-vs-half estimate)", fontsize=10)
ax.legend(fontsize=8.2, frameon=False, loc="upper left", ncol=2)
ax.grid(axis="y", alpha=0.25, lw=0.5)
fig.tight_layout()
plt.show()

**On top of the default-binned LightGBM, income's fine grid is worth +0.001906** (4.80 base SE,
positive in 5 of 5 folds). That is most of the +0.001999 that opening the histogram bought in
Part 2. Two unrelated instruments, a per-value lookup added from outside and a finer histogram
inside the trees, find the same missing piece, at the same size and in the same column. The
leak-free estimate is smaller, at +0.000434. That is the expected shape for an estimator working
from about an eighth of the rows at the same shrinkage.

**On top of the full-resolution LightGBM, it is worth +0.000497** (1.42 SE, under the bar). The
open histogram took most of it.

**On top of the blend, it is worth +0.000024, or +0.000047 by the leak-free estimate: 0.08 and
0.16 of the blend's SE.** The strong model already has income's fine structure. Its best members
carry quantisation digits and frequency encodings of income, according to the library's manifest,
and either is a way of handing a tree individual income values. That is my reading of why, and it
is an inference. The measurement is only that the correction adds nothing.

**Every other column, on every base, is at or below zero.** The largest single permutation-null
draw anywhere is +0.000001, so the probe does not invent gains out of cell structure. With the
recalibration in place, the in-fold slopes at full size run from 0.9886 to 1.0053, so every base
was already close to calibrated and the step barely mattered here. It mattered at 60,000 rows,
which is why it is in.

---

## Part 6: The pre-registration, scored

Before any AUC in this notebook was computed, I wrote down the theory (resolution is a one-column
effect here, income is the only column that pays past LightGBM's 255 bins, and a strong blend
already has it), a knee rule, a predicted shape for every column, and three falsifiers. The cell
below scores them from the variables above, so the verdict cannot drift from the numbers.

In [14]:
S = {s["col"]: s for s in SUMMARY}
OT = {(r["base"], r["col"]): r for r in ONTOP}
inc = "Annual_Income_USD"
others = [c for c in NUM if c != inc]

F1 = S[inc]["knee"] <= 256 or DEF_COST < BAR
F2 = [c for c in others if S[c]["knee"] > 64]
r3 = OT[("library blend", inc)]
F3 = r3["either"] >= CFG["BAR_SE"] * r3["base_se"]

print("FALSIFIERS (any one fires -> the theory is falsified)")
print(f"  F1  income knee {S[inc]['knee']:,} cells (needs > 256); default binning costs "
      f"{DEF_COST:+.6f} = {DEF_COST / SE_FULL:.2f} SE (needs >= 2)      -> "
      f"{'FIRED' if F1 else 'did not fire'}")
print(f"  F2  other columns with a knee above 64 cells: {F2 if F2 else 'none'}"
      f"{'':>30}-> {'FIRED' if F2 else 'did not fire'}")
print(f"  F3  income fine grid on top of the library blend: larger estimate {r3['either']:+.6f} = "
      f"{r3['either'] / r3['base_se']:.2f} blend SE (fires at >= 2)   -> {'FIRED' if F3 else 'did not fire'}")
print()

def hit(b):
    return "HIT " if b else "MISS"

SEC = [
    ("income single-column knee >= 1024", S[inc]["knee_alone"] >= 1024, f"{S[inc]['knee_alone']:,}"),
    ("income LightGBM knee >= 1024", S[inc]["knee"] >= 1024, f"{S[inc]['knee']:,}"),
    ("commute knee <= 64 (LightGBM)", S["Daily_Commute_km"]["knee"] <= 64, f"{S['Daily_Commute_km']['knee']:,}"),
    ("commute knee <= 64 (alone)", S["Daily_Commute_km"]["knee_alone"] <= 64,
     f"{S['Daily_Commute_km']['knee_alone']:,}"),
    ("Age knee <= 16", S["Age"]["knee"] <= 16, f"{S['Age']['knee']:,}"),
    ("Environmental_Concern_Level knee = 5", S["Environmental_Concern_Level"]["knee"] == 5,
     f"{S['Environmental_Concern_Level']['knee']:,}"),
    ("Environmental_Concern_Level has the largest column value",
     max(SUMMARY, key=lambda s: s["value"])["col"] == "Environmental_Concern_Level",
     max(SUMMARY, key=lambda s: s["value"])["col"]),
    ("Number_of_Cars_Owned: no measurable signal", not S["Number_of_Cars_Owned"]["has_signal"],
     f"value {S['Number_of_Cars_Owned']['value']:+.6f}"),
    ("Charging_Stations_Near_Home knee <= 8", S["Charging_Stations_Near_Home"]["knee"] <= 8,
     f"{S['Charging_Stations_Near_Home']['knee']:,}"),
    ("Charging_Stations_Near_Work knee <= 4", S["Charging_Stations_Near_Work"]["knee"] <= 4,
     f"{S['Charging_Stations_Near_Work']['knee']:,}"),
    ("income fine grid >= 2 SE on top of default LightGBM",
     OT[("default-binned LightGBM", inc)]["either"] >= CFG["BAR_SE"] * BASE_SE["default-binned LightGBM"],
     f"{OT[('default-binned LightGBM', inc)]['either']:+.6f}"),
    ("every other column < 2 SE on top of every base",
     all(OT[(b, c)]["either"] < CFG["BAR_SE"] * BASE_SE[b] for b in BASES for c in others),
     str([(b, c) for b in BASES for c in others
          if OT[(b, c)]["either"] >= CFG["BAR_SE"] * BASE_SE[b]] or "none above")),
]
print("SECONDARY PREDICTIONS (scored one by one; none of these falsifies the theory alone)")
for lbl, ok, got in SEC:
    print(f"  {hit(ok)}  {lbl:<58} measured: {got}")
print(f"\n  {sum(ok for _, ok, _ in SEC)} of {len(SEC)} secondary predictions hit.")

None of the three falsifiers fired, and 11 of 12 secondary predictions hit. That count flatters
me, and I would rather say so than let it stand. **Four of the hits are hollow.** I predicted
that commute, `Age` and the two charging-station counts would saturate at small cell counts, and
they "hit" because they carry no measurable signal in the model at all, so their knee is one cell
by the rule. I predicted a shape and got flatness. The one real miss is informative too: commute
**alone** keeps gaining to 201 cells, far past the 64 I predicted, and it still does not matter,
because the model does not use it.

---

## Part 7: What would make this wrong

**One model, one configuration.** The curves belong to this LightGBM as much as to the data. Its
`min_child_samples` means a leaf cannot isolate an income value held by fewer rows than that, so
the fine end of the income curve is bounded by it. A model with smaller leaves or more trees might
use more of income's grid, and the knee could move up. It would be hard to move it down.

**One partition, one seed for the curves.** Every curve point comes from the frozen library folds
and seed 42. I priced the seed directly (0.21 SE across three seeds), not the partition.

**The knee is only as fine as the grid.** The grid doubles, so "knee at 1,782 cells" means
somewhere between 926 and 1,782. The next coarser point is 3.00 SE below the best, and the knee
point is inside the bar.

**Opening the histogram changed three parameters at once:** `max_bin`, `min_data_in_bin` and
`bin_construct_sample_cnt`. The curves put the gain in income, the one column the default
coarsens that carries signal. I did not separate the three settings from each other.

**Cross-validation is not the leaderboard.** Every SE here is a fold SE on out-of-fold AUC. The
public board is scored on a much smaller sample and its standard error is larger. The
[ingredient-ablation notebook](https://www.kaggle.com/code/dariushafshar/s6e9-0-94585-1-of-4-ingredients-carries-it)
works through what that means for a gain of this size. Discount before you believe any of this
about a leaderboard.

**The on-top probe is a one-step correction.** It measures what a per-cell offset adds. A model
trained with the column's fine grid as a feature could extract more, so "the blend already has
it" means "a correction from outside adds nothing", not "no model could do better". Its two
estimators have opposite weaknesses, and they agree on the blend.

**The library's own caveats pass through to the blend.** Three library members early-stop on the
validation fold, which makes their OOF mildly optimistic, and two use frequency features computed
over train and test together. The library documents both.

**"The generator attaches a propensity to income values" is an inference.** It is how this data
behaves under two instruments here and a third in the
[income fine-structure notebook](https://www.kaggle.com/code/dariushafshar/s6e9-53-income-spikes-94-of-the-gain),
not a statement about code anyone has seen.

---

## Part 8: What to do with this

1. **If you run a GBM on the raw columns, open the histogram.** In LightGBM that is `max_bin`
   above income's distinct count, `min_data_in_bin=1`, and `bin_construct_sample_cnt` at least the
   training rows. The config cell here has the exact values, and they were worth +0.001999 on this
   model. XGBoost's `hist` method has a `max_bin` of its own that you can raise. scikit-learn's
   HistGradientBoosting cannot be opened past its default at all: it refuses any `max_bins` above
   the default.
2. **If you would rather keep the default histogram, give income a finer encoding.** A
   cross-fitted target encoding of income at a few thousand cells, or its value rounded to $10,
   carries the same signal. The
   [ingredient-ablation notebook](https://www.kaggle.com/code/dariushafshar/s6e9-0-94585-1-of-4-ingredients-carries-it)
   found that dual target encoding carries almost everything in a strong public recipe. In the
   terms of this notebook, target encoding a wide column is a way of buying it resolution.
3. **If you already have target encoding, digits or frequency features of income, stop here.**
   The strong blend in Part 5 gains nothing from the fine grid. Spend the time somewhere else.
4. **Do not spend time on the resolution of any other numeric column.** Commute, `Age`, cars owned
   and both charging-station counts carry nothing the model can measure at any resolution, and
   `Environmental_Concern_Level` needs its five levels and has no finer grid to find.

**The cell to fork is Part 3's loop.** Point `quantile_codes` and `run_lgb` at your own column,
keep the folds, and read the curve against two SE of the full-resolution model rather than against
zero. The resolution you need is where the curve enters that band, not where it looks flat to the
eye.

Related measurements on this board:
The [twins-and-shift notebook](https://www.kaggle.com/code/dariushafshar/s6e9-0-exact-twins-31-discrete-zero-shift)
found essentially no rows that repeat across all thirteen columns, while a large share repeat
across the discrete ones. The two wide
numerics are what make a row unique, and this notebook shows only one of them carries signal.
The [model-saturation notebook](https://www.kaggle.com/code/dariushafshar/s6e9-6-of-9-models-add-exactly-zero)
measures the plateau this all sits under. The interaction side of the same headroom is in
the [interaction census](https://www.kaggle.com/code/dariushafshar/s6e9-4-of-13-columns-hold-the-interaction).

If something here does not reproduce for you, say so in the comments with the cell and the
number. I would rather correct this than be quietly wrong in it.

In [15]:
HEAD = {
    "n_train": N, "smoke_rows": SMOKE_ROWS,
    "auc_full": round(AUC_FULL, 6), "se_full": round(SE_FULL, 6), "bar": round(BAR, 6),
    "auc_default_bins": round(float(FA_DEF.mean()), 6),
    "default_bin_cost": round(DEF_COST, 6), "default_bin_cost_se": round(DEF_COST / SE_FULL, 2),
    "seed_spread": round(float(seed_means.max() - seed_means.min()), 6),
    "blend_auc": round(float(FA_BLEND.mean()), 6), "blend_se": round(se(FA_BLEND), 6),
    "columns": {s["col"]: {"distinct": s["distinct"], "value": round(s["value"], 6),
                           "knee": s["knee"], "knee_auc": round(s["knee_auc"], 6),
                           "knee_alone": s["knee_alone"], "best_alone": round(s["best_alone"], 6),
                           "has_signal": bool(s["has_signal"])} for s in SUMMARY},
    "income_rounding": {str(r["step"]): [r["cells"], round(r["d"], 6), round(r["d"] / SE_FULL, 2)]
                        for r in ROUND},
    "on_top": {f"{r['base']} | {r['col']}": {"gain": round(r["gain"], 6), "se": round(r["se"], 6),
                                              "in_base_se": round(r["gain"] / r["base_se"], 2),
                                              "wins": r["wins"], "lam": r["lam"],
                                              "null": round(r["null_best"], 6),
                                              "half_gain": round(r["h_gain"], 6),
                                              "half_se": round(r["h_se"], 6)}
               for r in ONTOP},
    "falsified": bool(F1 or F2 or F3), "F1": bool(F1), "F2": F2, "F3": bool(F3),
    "secondary_hits": int(sum(ok for _, ok, _ in SEC)), "secondary_total": len(SEC),
    "wall_seconds": round(time.time() - T0, 1),
}
os.makedirs(WORK, exist_ok=True)
with open(os.path.join(WORK, "headline.json"), "w") as fh:
    json.dump(HEAD, fh, indent=1)
sub = pd.DataFrame({ID: test[ID], TARGET: TEST_FULL})
sub.to_csv(os.path.join(WORK, "submission.csv"), index=False)
print("written: headline.json, submission.csv (the full-resolution curve model's own test")
print("predictions, the mean of its five fold models; a by-product, not a tuned entry)")
print()
for k, v in HEAD.items():
    if k not in ("columns", "on_top", "income_rounding"):
        print(f"  {k:<22} {v}")
print(f"\n  total wall time {HEAD['wall_seconds'] / 60:.1f} min")
if SMOKE_ROWS:
    print("SMOKE_ROWS was set. Do NOT freeze a title from this run.")